In [1]:
import ee
import geemap
ee.Authenticate()
ee.Initialize()

In [2]:
Map = geemap.Map()

# Panama boundary
countries = ee.FeatureCollection("FAO/GAUL/2015/level0")
panama_fc = countries.filter(ee.Filter.eq("ADM0_NAME", "Panama"))
panama_geom = panama_fc.geometry()

Map.centerObject(panama_geom, 7)

In [3]:
districts = ee.FeatureCollection("projects/deforestation-panama/assets/Districts")

In [ ]:
wapor_aeti = (
    ee.ImageCollection("FAO/WAPOR/3/L1_AETI_D")
    .filterDate("2024-01-01", "2026-01-01")
    .filterBounds(panama_geom)
    .select("L1-AETI-D") 
)

def calculate_yearly_aeti(year):
    start = ee.Date.fromYMD(year, 1, 1)
    end = start.advance(1, "year")
    # Multiply by 0.1 scale factor to get total mm/year
    annual_sum = wapor_aeti.filterDate(start, end).sum().multiply(0.1).float().rename(f"aeti_{year}")
    return annual_sum

yearly_aeti = ee.Image.cat([calculate_yearly_aeti(yr) for yr in aeti_years])

### Refined code from up top

In [4]:
# Years of interest
aeti_years = [2024, 2025]

# WaPOR AETI daily dataset
wapor_aeti = (
    ee.ImageCollection("FAO/WAPOR/3/L1_AETI_D")
    .filterDate("2024-01-01", "2026-01-01")
    .filterBounds(panama_geom)
    .select("L1-AETI-D")
)

# Calculate annual cumulative AETI
def calculate_yearly_aeti(year):
    start = ee.Date.fromYMD(year, 1, 1)
    end = start.advance(1, "year")

    annual_sum = (
        wapor_aeti
        .filterDate(start, end)
        .sum()
        .multiply(0.1)
        .rename(f"aeti_{year}")
    )

    return annual_sum

# Create two-band image (2024 and 2025)
yearly_aeti = ee.Image.cat([
    calculate_yearly_aeti(yr) for yr in aeti_years
])

### Calculate annual AETI for 2024 and 2025

In [8]:
# Years of interest
aeti_years = [2024, 2025]

# Calculate annual cumulative AETI (mm/year)
def calculate_yearly_aeti(year):

    start = ee.Date.fromYMD(year, 1, 1)
    end = ee.Date.fromYMD(year + 1, 1, 1)

    collection = wapor_aeti.filterDate(start, end)

    # Convert each dekadal image to annual contribution
    def convert_to_mm(image):

        image = ee.Image(image).toFloat()

        date = ee.Date(image.get('system:time_start'))
        day = date.get('day')

        month_end = ee.Date.fromYMD(
            date.get('year'),
            date.get('month'),
            1
        ).advance(1, 'month')

        days = ee.Number(
            ee.Algorithms.If(
                day.lte(10), 10,
                ee.Algorithms.If(
                    day.lte(20), 10,
                    month_end.difference(date, 'day')
                )
            )
        )

        return (
            image
            .multiply(0.1)
            .multiply(days)
            .toFloat()
            .copyProperties(image, ['system:time_start'])
        )

    # Sum dekadal contributions
    annual_total = (
        collection
        .map(convert_to_mm)
        .sum()
        .toFloat()
        .rename(f'aeti_{year}')
    )

    return annual_total


# Recreate annual images
yearly_aeti = ee.Image.cat([
    calculate_yearly_aeti(yr) for yr in aeti_years
])

### Calculate mean AETI within each district

In [9]:
# Calculate mean AETI for each district
district_aeti = yearly_aeti.reduceRegions(
    collection=districts,
    reducer=ee.Reducer.mean(),
    scale=326.13,
    tileScale=4
)

# Keep only district name and AETI values
district_aeti = district_aeti.select([
    'Distrito',
    'aeti_2024',
    'aeti_2025'
])

### Create the dataframe

In [10]:
import pandas as pd

# Retrieve district features
features_info = district_aeti.getInfo()['features']

rows = []

for f in features_info:

    props = f['properties']

    aeti_2024 = props.get('aeti_2024')
    aeti_2025 = props.get('aeti_2025')

    # Calculate change between years
    if aeti_2024 is not None and aeti_2025 is not None:
        change = aeti_2025 - aeti_2024
    else:
        change = None

    row = {
        'DISTRICT': props.get('Distrito'),
        'AETI_2024_MM': round(aeti_2024, 2) if aeti_2024 is not None else None,
        'AETI_2025_MM': round(aeti_2025, 2) if aeti_2025 is not None else None,
        'AETI_CHANGE_MM': round(change, 2) if change is not None else None
    }

    rows.append(row)

# Create dataframe
df = pd.DataFrame(rows)

# Sort alphabetically by district
df = df.sort_values(
    by='DISTRICT'
).reset_index(drop=True)

# Display dataframe
print(df)

         DISTRICT  AETI_2024_MM  AETI_2025_MM  AETI_CHANGE_MM
0       Aguadulce        762.76        887.73          124.97
1          Alanje        941.94       1132.49          190.56
2       Almirante       1020.70       1011.33           -9.37
3           Antón        869.96        977.12          107.17
4        Arraiján        948.78       1003.56           54.78
..            ...           ...           ...             ...
77         Taboga           NaN           NaN             NaN
78  Tierras Altas        967.47        986.55           19.08
79           Tolé       1072.48       1151.56           79.08
80         Tonosí        850.74        900.66           49.92
81          Ñürün        858.68        904.01           45.33

[82 rows x 4 columns]


In [ ]:
df.to_csv(
    'Panama_AETI_Change_by_District_2024_2025.csv',
    index=False
)